# CrossDocked pocket embeddings with ProtT5

This notebook converts the raw CrossDocked pocket/SMILES CSV into the residue-level ProtT5 assets required before tokenization.

It produces:

- `per_residue_pack.npy`: a memory-mapped `(sum(unique_sequence_lengths), 1024)` float16 array.
- `per_residue_index.parquet`: one row per pocket/ligand pair with `start`, `length`, `seq_pocket`, and `smiles`.

After running the notebook, use `scripts/tokenize_dataset.py` to add SELFIES and `token_ids` to the final finetuning Parquet. The NPY format supports true memory-mapped reads during training. Run the cells in order.


In [ ]:
# Install notebook dependencies. PyTorch is preinstalled on GPU Colab runtimes.
!pip -q install pandas pyarrow transformers sentencepiece tqdm


In [ ]:
from pathlib import Path

# Change this only when the repository is mounted elsewhere.
REPO_ROOT = Path("/content/PockLigGPT_official")
RAW_CSV_PATH = REPO_ROOT / "datasets/raw/crossdocked/merged_pocket_smiles.csv"
OUTPUT_DIR = REPO_ROOT / "datasets/processed/crossdocked"
INDEX_PATH = OUTPUT_DIR / "per_residue_index.parquet"
EMBEDDINGS_PATH = OUTPUT_DIR / "per_residue_pack.npy"

MODEL_NAME = "Rostlab/prot_t5_xl_uniref50"
BATCH_SIZE = 4
EMBEDDING_DTYPE = "float16"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Prepare the source table

The raw CSV contains one `pocket_smiles` column in the form `<one-letter pocket sequence>_<SMILES>`. Pocket identifiers are deterministic hashes of the normalized sequence, so rerunning the notebook preserves the same IDs. Embeddings are computed once per unique pocket sequence and then shared by all ligands associated with that pocket.


In [ ]:
import hashlib
import re

import numpy as np
import pandas as pd

VALID_AA = re.compile(r"^[ACDEFGHIKLMNPQRSTVWYX]+$")
REPLACED_AA = re.compile(r"[UZOB]")

def normalize_sequence(value: str) -> str:
    sequence = str(value).strip().upper().replace(" ", "")
    sequence = REPLACED_AA.sub("X", sequence)
    if not sequence or not VALID_AA.fullmatch(sequence):
        raise ValueError(f"Invalid pocket sequence: {value!r}")
    return sequence

def pocket_id(sequence: str) -> str:
    digest = hashlib.sha256(sequence.encode("ascii")).hexdigest()[:16]
    return f"p_{digest}"

raw_df = pd.read_csv(RAW_CSV_PATH)
if "pocket_smiles" not in raw_df.columns:
    raise ValueError("The raw CSV must contain a 'pocket_smiles' column.")

split_values = raw_df["pocket_smiles"].astype(str).str.split("_", n=1, expand=True)
if split_values.shape[1] != 2:
    raise ValueError("Every pocket_smiles value must contain an underscore separator.")

pairs_df = pd.DataFrame({
    "seq_pocket": split_values[0].map(normalize_sequence),
    "smiles": split_values[1].str.strip(),
})
pairs_df = pairs_df[pairs_df["smiles"].ne("")].reset_index(drop=True)
pairs_df["pocket_id"] = pairs_df["seq_pocket"].map(pocket_id)

pockets_df = (
    pairs_df[["pocket_id", "seq_pocket"]]
    .drop_duplicates(subset="seq_pocket")
    .reset_index(drop=True)
)
pockets_df["length"] = pockets_df["seq_pocket"].str.len().astype("int64")

print(f"Ligand rows: {len(pairs_df):,}")
print(f"Unique pocket sequences: {len(pockets_df):,}")
print(f"Total residues: {int(pockets_df['length'].sum()):,}")
pockets_df.head()


## 2. Load ProtT5

ProtT5 expects amino acids separated by spaces. Special tokens are disabled so the number of model positions equals the number of residues exactly. Ambiguous residues `U`, `Z`, `O`, and `B` were normalized to `X` in the previous cell.


In [ ]:
import torch
from transformers import T5EncoderModel, T5Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)
model = T5EncoderModel.from_pretrained(MODEL_NAME)
model = model.half().to(device) if device.type == "cuda" else model.float().to(device)
model.eval()

embedding_dim = int(model.config.d_model)
if embedding_dim != 1024:
    raise ValueError(f"Expected ProtT5 embedding dimension 1024, got {embedding_dim}.")
print("Device:", device)
print("Embedding dimension:", embedding_dim)


In [ ]:
def encode_batch(sequences):
    spaced = [" ".join(sequence) for sequence in sequences]
    encoded = tokenizer(
        spaced,
        add_special_tokens=False,
        padding=True,
        return_tensors="pt",
    )

    expected_lengths = torch.tensor([len(sequence) for sequence in sequences])
    observed_lengths = encoded["attention_mask"].sum(dim=1).cpu()
    if not torch.equal(expected_lengths, observed_lengths):
        raise ValueError(
            f"ProtT5 token/residue mismatch: expected {expected_lengths.tolist()}, "
            f"got {observed_lengths.tolist()}"
        )

    encoded = {name: tensor.to(device) for name, tensor in encoded.items()}
    with torch.no_grad():
        hidden = model(**encoded).last_hidden_state
    return hidden, expected_lengths.tolist()

# Fast alignment check before processing the complete dataset.
example_sequences = pockets_df["seq_pocket"].head(2).tolist()
example_hidden, example_lengths = encode_batch(example_sequences)
print("Alignment check:", example_lengths, tuple(example_hidden.shape))


## 3. Write the residue embedding stack and index

The output array is allocated once as a valid NPY memmap. Each index row records the half-open interval `[start, start + length)` belonging to one pocket sequence. Re-running this cell overwrites the generated files.


In [ ]:
from tqdm.auto import tqdm

total_residues = int(pockets_df["length"].sum())
emb_stack = np.lib.format.open_memmap(
    EMBEDDINGS_PATH,
    mode="w+",
    dtype=EMBEDDING_DTYPE,
    shape=(total_residues, embedding_dim),
)

index_rows = []
cursor = 0

for batch_start in tqdm(range(0, len(pockets_df), BATCH_SIZE)):
    batch_df = pockets_df.iloc[batch_start:batch_start + BATCH_SIZE]
    sequences = batch_df["seq_pocket"].tolist()
    hidden, lengths = encode_batch(sequences)

    for batch_index, (_, row) in enumerate(batch_df.iterrows()):
        length = lengths[batch_index]
        residue_embeddings = (
            hidden[batch_index, :length]
            .float()
            .cpu()
            .numpy()
            .astype(EMBEDDING_DTYPE, copy=False)
        )
        emb_stack[cursor:cursor + length] = residue_embeddings
        index_rows.append({
            "pocket_id": row["pocket_id"],
            "seq_pocket": row["seq_pocket"],
            "start": cursor,
            "length": length,
        })
        cursor += length

    del hidden
    if device.type == "cuda":
        torch.cuda.empty_cache()

emb_stack.flush()
del emb_stack

if cursor != total_residues:
    raise ValueError(f"Wrote {cursor} residues, expected {total_residues}.")

unique_index_df = pd.DataFrame(index_rows)
index_df = pairs_df.merge(
    unique_index_df,
    on=["pocket_id", "seq_pocket"],
    how="left",
    validate="many_to_one",
)
index_df = index_df[["pocket_id", "start", "length", "seq_pocket", "smiles"]]
index_df.to_parquet(INDEX_PATH, index=False)
print("Embedding stack:", EMBEDDINGS_PATH)
print("Embedding index:", INDEX_PATH)


## 4. Tokenize the pocket/ligand pairs

Return to the repository root and run the project tokenization command. It reads the embedding index produced above and writes the final Parquet consumed by finetune 2.


In [ ]:
# Run this from a terminal in the repository root, not inside Python.
# python scripts/tokenize_dataset.py --config config/tokenization/crossdocked.yaml


## 5. Validate the embedding assets

The index may contain repeated offsets when several ligands share the same pocket sequence. That is expected: the embedding stack stores each unique sequence once.


In [ ]:
index_df = pd.read_parquet(INDEX_PATH)
emb_stack = np.load(EMBEDDINGS_PATH, mmap_mode="r")

required_columns = {"pocket_id", "start", "length", "seq_pocket", "smiles"}
missing = required_columns - set(index_df.columns)
if missing:
    raise ValueError(f"Embedding index is missing columns: {sorted(missing)}")
if not index_df["length"].eq(index_df["seq_pocket"].str.len()).all():
    raise ValueError("Embedding lengths do not match the pocket sequences.")
if int((index_df["start"] + index_df["length"]).max()) > len(emb_stack):
    raise ValueError("At least one embedding interval falls outside the stack.")
if emb_stack.ndim != 2 or emb_stack.shape[1] != 1024:
    raise ValueError(f"Unexpected embedding shape: {emb_stack.shape}")

print("Index rows:", len(index_df))
print("Unique embedding intervals:", index_df[["start", "length"]].drop_duplicates().shape[0])
print("Embedding stack:", emb_stack.shape, emb_stack.dtype)
